In [11]:
!pip install scikit-learn keras tensorflow

     ---------------------------------------- 0.0/57.5 kB ? eta -:--:--
     ---------------------------------------- 57.5/57.5 kB 1.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/65.5 kB ? eta -:--:--
     ---------------------------------------- 65.5/65.5 kB 3.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
     --------- ------------------------------ 0.4/1.5 MB 11.2 MB/s eta 0:00:01
     ------------------- -------------------- 0.7/1.5 MB 9.3 MB/s eta 0:00:01
     --------------------- ------------------ 0.8/1.5 MB 7.1 MB/s eta 0:00:01
     ---------------------------------------- 1.5/1.5 MB 9.5 MB/s eta 0:00:00
  Using cached requests-2.31.0-py3-none-any.whl.metadata (4.6 kB)
     ---------------------------------------- 0.0/181.3 kB ? eta -:--:--
     ------------------------------------- 181.3/181.3 kB 11.4 MB/s eta 0:00:00
  Using cached charset_normalizer-3.3.2-cp310-cp310-win_amd64.whl.metadata (34 kB)
  Using ca

In [12]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences

In [13]:
# Load the dataset
df = pd.read_csv('df_start.tsv')

# Convert timestamps to pandas datetime format
df['timestamp'] = pd.to_datetime(df['timestamp'])

### Time bins

In [14]:
# Divide the day into 8 time bins
time_bins = pd.cut(df['timestamp'].dt.hour, bins=[0, 3, 6, 9, 12, 15, 18, 21, 24], labels=False, right=False)

# Add time bins to the dataframe
df['time_bin'] = time_bins

### Encode user and app names

In [15]:
# Encode user and app names to numeric ids
user_encoder = LabelEncoder()
app_encoder = LabelEncoder()
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['app_name'] = app_encoder.fit_transform(df['app_name'])

###  Sort by user and timestamp

In [16]:
df = df.sort_values(by=['user_id', 'timestamp'])

### Group the apps used by sequences

In [17]:
grouped = df.groupby('user_id')['app_name'].apply(list)

In [18]:
# Create fixed-size sequences
max_sequence_length = 10  # You can set this to the desired sequence length
sequences = pad_sequences(grouped, maxlen=max_sequence_length, padding='pre')

In [20]:
len(sequences)

292

### NeuSA

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate

### Hyperparameters

In [22]:
embedding_dim = 50  # size of the embedding vectors
num_users = len(user_encoder.classes_)  # number of unique users
num_apps = len(app_encoder.classes_)  # number of unique apps
sequence_length = max_sequence_length

### User and app input layers

In [23]:
user_input = Input(shape=(1,), name='user_input')
app_input = Input(shape=(sequence_length,), name='app_input')
context_input = Input(shape=(sequence_length, embedding_dim), name='context_input')

### Embedding layers

In [27]:
user_embedding = Embedding(num_users, embedding_dim, input_length=1, name='user_embedding')(user_input)
app_embedding = Embedding(num_apps, embedding_dim, input_length=sequence_length, name='app_embedding')(app_input)

### Flatten the user embedding to remove the unnecessary dimension

In [25]:
user_embedding = tf.keras.layers.Flatten()(user_embedding)

### Concatenate the embeddings with context

In [28]:
concatenated = Concatenate()([user_embedding, app_embedding, context_input])

ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 1, 50), (None, 10, 50), (None, 10, 50)]

### LSTM layer

In [29]:
lstm_out = LSTM(128)(concatenated)

NameError: name 'concatenated' is not defined

### Fully connected layers

In [30]:
dense1 = Dense(256, activation='relu')(lstm_out)
dropout1 = Dropout(0.5)(dense1)
dense2 = Dense(128, activation='relu')(dropout1)
dropout2 = Dropout(0.5)(dense2)


NameError: name 'lstm_out' is not defined

### Output layer

In [ ]:
output = Dense(num_apps, activation='softmax')(dropout2)

### Create the model

In [ ]:
model = Model(inputs=[user_input, app_input, context_input], outputs=output)

### Compile the model

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

### Model summary 

In [ ]:
model.summary()

### Train the model

In [ ]:
# Split the data into training and validation sets
# You can use sklearn's train_test_split function for this purpose

# Train the model
history = model.fit([user_ids, app_sequences, context_sequences], 
                    app_labels, 
                    epochs=10, 
                    batch_size=32, 
                    validation_split=0.1)

### Evaluating the model

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate([test_user_ids, test_app_sequences, test_context_sequences], 
                                          test_app_labels)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")
